# Your Voice, Your Song: entrenamiento GPU privado
Clona solo el codigo. Las muestras se importan desde un ZIP privado, nunca desde GitHub. Al terminar se generan un modelo de inferencia sin grabaciones y un paquete privado para futuros ajustes. No inicia una web publica ni tuneles. Usa paquetes propios o de confianza.

Selecciona un runtime GPU. Recomendacion inicial: A100 para priorizar tiempo, L4 para un ajuste pequeno, T4 si es lo disponible. G4/Blackwell usa CUDA 12.8; la celda de instalacion verifica una operacion real CUDA. La GPU no garantiza mayor parecido vocal.


In [ ]:
from pathlib import Path
import subprocess, sys, json, shutil

REPO_URL = ""  # https://github.com/TU_USUARIO/TU_REPOSITORIO.git
MODE = "fine_tune"  # initial para el primer entrenamiento
EPOCHS = 50 if MODE == "fine_tune" else 200
BATCH_SIZE = 4  # Conservador para empezar; reduce a 2 si falta VRAM
PROJECT = Path("/content/your-voice-your-song")
assert REPO_URL.startswith("https://github.com/") and "@" not in REPO_URL, "Introduce la URL GitHub, sin tokens"
assert MODE in {"fine_tune", "initial"}
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
else:
    print("Se conserva el checkout existente; no se reinician ni sobrescriben datos.")
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
# Solo ffmpeg/libsndfile son dependencias del sistema. Python, CUDA wheels y caches viven en el proyecto.
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libsndfile1"], check=True)
subprocess.run(["bash", "install_colab.sh", MODE], cwd=PROJECT, check=True)
PYTHON = str(PROJECT / "backend/.venv-vc/bin/python")


## Importar el paquete privado
En la web local, descarga **Paquete privado para Colab** despues de aceptar tus nuevas tomas. Este ZIP contiene datos biometricos y checkpoints: no lo publiques ni lo anadas a Git. Subirlo aqui lo transmite a tu runtime de Google. Para fine-tuning necesitas G/D y un dataset actualizado respecto al modelo; el ZIP pequeno de inferencia no sirve para entrenar.

Para evitar subir un archivo grande por el navegador, tambien puedes montar tu propio Drive manualmente y colocar su ruta en `PRIVATE_ZIP`. Las importaciones exigen un perfil nuevo y conservan los holdouts originales.


In [ ]:
PRIVATE_ZIP = ""  # Opcional: /content/drive/MyDrive/voice-training-private.zip
if not PRIVATE_ZIP:
    from google.colab import files
    incoming = files.upload()
    assert len(incoming) == 1, "Sube un solo paquete privado"
    name = next(iter(incoming))
    destination = PROJECT / "private" / "voice-training-private.zip"
    destination.parent.mkdir(exist_ok=True)
    shutil.move(name, destination)
    incoming.clear()
    PRIVATE_ZIP = str(destination)
subprocess.run([PYTHON, "backend/portable.py", "import-training", PRIVATE_ZIP], cwd=PROJECT, check=True)


In [ ]:
# Entrenamiento real CUDA; sin fallback accidental a CPU. Los logs quedan dentro del proyecto.
subprocess.run([PYTHON, "backend/portable.py", "train", "--mode", MODE, "--epochs", str(EPOCHS),
                "--batch-size", str(BATCH_SIZE), "--require-cuda"], cwd=PROJECT, check=True)


## Guardar los resultados
Colab es temporal: descarga los resultados antes de desconectar. `voice-model.zip` contiene pesos e indice, sin audio, pero sigue siendo tu identidad vocal y debe mantenerse privado. `voice-training-private.zip` incluye muestras y G/D para el siguiente ajuste. Si montaste Drive, puedes elegir una carpeta privada en la siguiente celda para conservar ambos. Si la sesion se interrumpe antes de terminar, conserva tu paquete de entrada para volver a ejecutar el ajuste; no se promete reanudacion exacta del paso interrumpido.

En la web local marca la procedencia de confianza, importa `voice-model.zip`, genera una prueba y apruebala antes de convertir canciones. La web no necesita las grabaciones de entrenamiento para inferencia.


In [ ]:
SAVE_DIR = ""  # Opcional: carpeta privada de tu Drive previamente montado
outputs = PROJECT / "exports"
assert (outputs / "voice-model.zip").is_file()
if SAVE_DIR:
    destination = Path(SAVE_DIR)
    destination.mkdir(parents=True, exist_ok=True)
    for name in ("voice-model.zip", "voice-training-private.zip"):
        shutil.copy2(outputs / name, destination / name)
    print("Resultados guardados en", destination)
else:
    from google.colab import files
    files.download(str(outputs / "voice-model.zip"))
    print("Descarga tambien exports/voice-training-private.zip desde el explorador para futuros ajustes.")
